# OOV: word2vec vs. GloVe

Uses the loading and preprocessing from the other static solution - simply compares oov counts and lookup metrics.

In [1]:
%pip install -q datasets gensim nltk

Note: you may need to restart the kernel to use updated packages.


In [2]:
from collections import Counter

import gensim.downloader as api
import nltk
from datasets import load_dataset

nltk.download("punkt_tab", quiet=True)

True

In [3]:
EMBEDDINGS = {
    "w2v": "word2vec-google-news-300",
    "glove": "glove-wiki-gigaword-300",
}

In [4]:
dataset = load_dataset("amarfurt/art-nlp-hs26")

train = dataset["train"]
valid = dataset["validation"]
test = dataset["test"]

print(f"train: {len(train)}, valid: {len(valid)}, test: {len(test)}")

train: 16801, valid: 1000, test: 1532


In [5]:
def preprocess(text):
    tokens = nltk.word_tokenize(text)
    return [token for token in tokens if any(character.isalnum() for character in token)]

In [6]:
TEXT_FIELDS = ["observation_1", "observation_2", "hypothesis_1", "hypothesis_2"]

def preprocess_example(example):
    processed = {field: preprocess(example[field]) for field in TEXT_FIELDS}
    processed["target"] = example["label"] - 1
    return processed


def preprocess_split(split):
    return split.map(preprocess_example)

In [7]:
train_tokens = preprocess_split(train)
valid_tokens = preprocess_split(valid)
test_tokens = preprocess_split(test)

In [8]:
embeddings = {name: api.load(model_id) for name, model_id in EMBEDDINGS.items()}

for name, kv in embeddings.items():
    print(f"{name}: {len(kv.index_to_key)} words, {kv.vector_size} dimensions")

[===-----------------------------------------------] 6.3% 23.5/376.1MB downloaded

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



[==================================================] 100.0% 376.1/376.1MB downloaded
w2v: 3000000 words, 300 dimensions
glove: 400000 words, 300 dimensions


### Lookup

In [12]:
sentences = [sentence for split in (train_tokens, valid_tokens, test_tokens) for field in TEXT_FIELDS for sentence in split[field]]
token_counts = Counter(token for sentence in sentences for token in sentence)
total_tokens = sum(token_counts.values())

print(f"sentences: {len(sentences)}, tokens: {total_tokens}, types: {len(token_counts)}")


def lookup(kv, token):
    if token in kv:
        return "exact"
    if token.lower() in kv:
        return "lowercase"
    return "oov"

sentences: 77332, tokens: 670395, types: 19041


In [10]:
metrics = {
    "oov/tokens": total_tokens,
    "oov/types": len(token_counts),
}
oov_types = {}

for name, kv in embeddings.items():
    status = {token: lookup(kv, token) for token in token_counts}
    occurrences = Counter()
    for token, count in token_counts.items():
        occurrences[status[token]] += count
    oov_types[name] = {token for token, result in status.items() if result == "oov"}

    metrics.update({
        f"oov/{name}/vocab_size": len(kv.index_to_key),
        f"oov/{name}/hits_exact": occurrences["exact"],
        f"oov/{name}/hits_lowercase": occurrences["lowercase"],
        f"oov/{name}/occurrences": occurrences["oov"],
        f"oov/{name}/rate": occurrences["oov"] / total_tokens,
        f"oov/{name}/unique_types": len(oov_types[name]),
        f"oov/{name}/type_rate": len(oov_types[name]) / len(token_counts),
        f"oov/{name}/empty_sentences": sum(all(status[token] == "oov" for token in sentence) for sentence in sentences),
    })

oov_both = oov_types["w2v"] & oov_types["glove"]
metrics.update({
    "oov/both/unique_types": len(oov_both),
    "oov/both/occurrences": sum(token_counts[token] for token in oov_both),
    "oov/only_w2v/unique_types": len(oov_types["w2v"] - oov_types["glove"]),
    "oov/only_glove/unique_types": len(oov_types["glove"] - oov_types["w2v"]),
})

for key, value in metrics.items():
    print(f"{key:32} {value:.4f}" if isinstance(value, float) else f"{key:32} {value}")

oov/tokens                       670395
oov/types                        19041
oov/w2v/vocab_size               3000000
oov/w2v/hits_exact               591672
oov/w2v/hits_lowercase           18
oov/w2v/occurrences              78705
oov/w2v/rate                     0.1174
oov/w2v/unique_types             1142
oov/w2v/type_rate                0.0600
oov/w2v/empty_sentences          6
oov/glove/vocab_size             400000
oov/glove/hits_exact             575426
oov/glove/hits_lowercase         93825
oov/glove/occurrences            1144
oov/glove/rate                   0.0017
oov/glove/unique_types           861
oov/glove/type_rate              0.0452
oov/glove/empty_sentences        7
oov/both/unique_types            633
oov/both/occurrences             792
oov/only_w2v/unique_types        509
oov/only_glove/unique_types      228


In [11]:
for name, types in {**oov_types, "both": oov_both}.items():
    top = sorted(types, key=token_counts.get, reverse=True)[:20]
    print(f"{name}: {[(token, token_counts[token]) for token in top]}")

w2v: [('to', 26514), ('a', 22768), ('and', 15023), ('of', 6595), ("'s", 5582), ('20', 75), ('10', 72), ('didnt', 58), ('cancelled', 48), ('100', 44), ('50', 40), ('30', 39), ('co-worker', 30), ('15', 25), ('12', 24), ('18', 21), ('16', 17), ('200', 16), ('co-workers', 13), ('21', 11)]
glove: [('selfie', 11), ('mTurk', 8), ('Zaiden', 7), ('Mturk', 6), ('A+', 6), ('mturk', 6), ('selfies', 6), ('Antonela', 5), ('dogwhistle', 5), ('Fariah', 5), ('Benisha', 4), ('Marisse', 4), ('Myah', 4), ('Loraline', 4), ('Shaylee', 4), ('Lamay', 4), ('farted', 4), ('Meisha', 4), ('saded', 4), ('Lillah', 4)]
both: [('selfie', 11), ('mTurk', 8), ('Mturk', 6), ('A+', 6), ('mturk', 6), ('selfies', 6), ('Antonela', 5), ('dogwhistle', 5), ('Fariah', 5), ('Benisha', 4), ('Marisse', 4), ('Loraline', 4), ('saded', 4), ('inessentialed', 4), ('Shalil', 4), ('BunnBunn', 4), ('Padfoot', 4), ('rubix', 4), ('crapes', 3), ('Tuie', 3)]
